# 📄 Varalaxmi Jangili — Document Analyst
## Multimodal Crime / Incident Report Analyzer

**Data Type:** Police reports / official incident documents (PDF)  
**Objective:** Extract structured info from PDFs → CSV with Report_ID, Department, Doc_Type, Date, Program, Key_Detail

### Pipeline:
1. Load **all PDFs** from `pdf_data/` folder
2. Extract text per page — **auto-detect scanned pages → OCR fallback**
3. Detect **document boundaries** → split into individual cases
4. Extract structured fields using **spaCy NER + Regex + keyword classification**
5. Classify **document type** and **incident type** for each case
6. Generate clean **Key_Detail summary** (boilerplate stripped)
7. Output structured CSV with **Incident_ID** for integration

### Dataset:
- **Source:** [Arkansas Police 1033 Training Proposals — MuckRock (FOIA)](https://www.muckrock.com/foi/arkansas-114/arkansas-police-departments-1033-training-plan-proposals-20493/#file-52365)
- **File:** `LESO2.pdf` — Multi-page PDF with training plans, SOPs, memoranda, and scanned documents from multiple Arkansas police departments


In [1]:
# =============================================
# CELL 1: Install Dependencies
# =============================================
!pip install -q pdfplumber pymupdf pytesseract spacy pandas Pillow
!python -m spacy download en_core_web_sm
!apt-get install -y -q tesseract-ocr > /dev/null 2>&1

print("\n✅ All dependencies installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 61.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 119.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

✅ All dependencies installed!


In [2]:
# =============================================
# CELL 2: Import Libraries
# =============================================
import pdfplumber
import fitz  # PyMuPDF
import pytesseract
import spacy
import pandas as pd
import os
import re
import warnings
warnings.filterwarnings('ignore')

from PIL import Image

nlp = spacy.load("en_core_web_sm")
os.makedirs("pdf_data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print("✅ All libraries loaded!")

✅ All libraries loaded!


In [3]:
# =============================================
# CELL 3: Load All PDFs into pdf_data/
# =============================================
import shutil

if os.path.exists("pdf_data"):
    shutil.rmtree("pdf_data")
os.makedirs("pdf_data")

# Copy all available PDFs (add more here if available)
shutil.copy("/content/LESO2.pdf", "pdf_data/LESO2.pdf")

# Loop through and display all loaded PDFs
pdf_files = sorted([f for f in os.listdir("pdf_data") if f.endswith(".pdf")])
print(f"📁 Loaded {len(pdf_files)} PDF file(s):\n")
for f in pdf_files:
    filepath = os.path.join("pdf_data", f)
    doc = fitz.open(filepath)
    print(f"  📄 {f} — {len(doc)} pages, {os.path.getsize(filepath)//1024} KB")
    doc.close()

print(f"\n✅ All PDFs ready for processing!")

📁 Loaded 1 PDF file(s):

  📄 LESO2.pdf — 75 pages, 8161 KB

✅ All PDFs ready for processing!


In [4]:
# =============================================
# CELL 4: Text Extraction Functions (with OCR Fallback)
# =============================================
# Strategy per page:
#   1. Try pdfplumber  (best for structured/table text)
#   2. Try PyMuPDF     (fast general-purpose extraction)
#   3. If both return < 50 chars → page is scanned → OCR at 300 DPI

def extract_text_pdfplumber(pdf_path, page_num):
    """Extract text from a single page using pdfplumber."""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            if page_num < len(pdf.pages):
                text = pdf.pages[page_num].extract_text()
                return text.strip() if text else ""
    except Exception:
        pass
    return ""


def extract_text_pymupdf(pdf_path, page_num):
    """Extract text from a single page using PyMuPDF (fitz)."""
    try:
        doc = fitz.open(pdf_path)
        text = doc[page_num].get_text().strip()
        doc.close()
        return text
    except Exception:
        return ""


def extract_text_ocr(pdf_path, page_num):
    """OCR fallback: render page as 300 DPI image → pytesseract."""
    try:
        doc = fitz.open(pdf_path)
        page = doc[page_num]
        mat = fitz.Matrix(300/72, 300/72)  # 300 DPI rendering
        pix = page.get_pixmap(matrix=mat)
        img_path = f"/tmp/ocr_page_{page_num}.png"
        pix.save(img_path)
        doc.close()

        img = Image.open(img_path)
        text = pytesseract.image_to_string(img).strip()
        os.remove(img_path)
        return text
    except Exception as e:
        print(f"    ⚠️ OCR error on page {page_num + 1}: {e}")
        return ""


def smart_extract_page(pdf_path, page_num):
    """Smart extraction with automatic OCR fallback for scanned pages."""
    # Step 1: Try pdfplumber
    text = extract_text_pdfplumber(pdf_path, page_num)
    if len(text.strip()) >= 50:
        return text, "pdfplumber"

    # Step 2: Try PyMuPDF
    text = extract_text_pymupdf(pdf_path, page_num)
    if len(text.strip()) >= 50:
        return text, "PyMuPDF"

    # Step 3: Text too short → scanned page → OCR fallback
    print(f"    🔍 Page {page_num+1}: Only {len(text.strip())} chars extracted → Scanned page detected → Using OCR fallback...")
    ocr_text = extract_text_ocr(pdf_path, page_num)
    if len(ocr_text.strip()) > len(text.strip()):
        return ocr_text, "OCR"
    return text, "OCR (low yield)"


print("✅ Extraction functions ready (pdfplumber → PyMuPDF → OCR fallback)")

✅ Extraction functions ready (pdfplumber → PyMuPDF → OCR fallback)


In [5]:
# =============================================
# CELL 5: Extract Text from All Pages in All PDFs
# =============================================

all_pages = []

for pdf_file in sorted(os.listdir("pdf_data")):
    if not pdf_file.endswith(".pdf"):
        continue

    pdf_path = os.path.join("pdf_data", pdf_file)
    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    doc.close()

    print(f"\n📄 Processing: {pdf_file} ({total_pages} pages)")
    print("-" * 55)

    for pg in range(total_pages):
        text, method = smart_extract_page(pdf_path, pg)

        all_pages.append({
            "source_file": pdf_file,
            "page_num": pg + 1,
            "text": text,
            "method": method,
        })

        icon = "🔍" if "OCR" in method else "📝"
        preview = text[:60].replace("\n", " ").strip()
        print(f"  Page {pg+1:2d} {icon} [{method:12s}] ({len(text):>5,} chars): {preview}...")

text_count = sum(1 for p in all_pages if "OCR" not in p["method"])
ocr_count  = sum(1 for p in all_pages if "OCR" in p["method"])
print(f"\n{'='*55}")
print(f"✅ Total pages extracted: {len(all_pages)}")
print(f"   📝 Text-based: {text_count}  |  🔍 Scanned (OCR): {ocr_count}")
print(f"{'='*55}")


📄 Processing: LESO2.pdf (75 pages)
-------------------------------------------------------
    🔍 Page 1: Only 0 chars extracted → Scanned page detected → Using OCR fallback...
  Page  1 🔍 [OCR         ] (1,936 chars): May 26, 2015  Sheriff Kelley Cradduck Major Nathan Atchison...
    🔍 Page 2: Only 0 chars extracted → Scanned page detected → Using OCR fallback...
  Page  2 🔍 [OCR         ] (1,197 chars): ae hy  Sheriff Kelley Cradduck . Major Nathan Atchison Chief...
    🔍 Page 3: Only 0 chars extracted → Scanned page detected → Using OCR fallback...
  Page  3 🔍 [OCR         ] (1,869 chars): Benton County Sheriff’s Office POLICIES AND PROCEDURES  SUBJ...
    🔍 Page 4: Only 0 chars extracted → Scanned page detected → Using OCR fallback...
  Page  4 🔍 [OCR         ] (2,386 chars): Vi.  Monthly inspection, to include running the motor while...
    🔍 Page 5: Only 0 chars extracted → Scanned page detected → Using OCR fallback...
  Page  5 🔍 [OCR         ] (  985 chars): I.  Ill.  ARMORED R

In [6]:
# =============================================
# CELL 6: Detect Document Boundaries & Group into Cases
# =============================================
# The PDF contains multiple documents back-to-back:
#   memoranda, SOPs, training plans, cover sheets, scanned records, etc.
# We detect where each NEW document begins and group pages into cases.

DOCUMENT_START_PATTERNS = [
    r'(?i)to\s*:\s*whom\s*it\s*may\s*concern',
    r'(?i)standard\s+operating\s+procedure',
    r'(?i)cover\s+sheet',
    r'(?i)course\s+summary',
    r'(?i)course\s+outline',
    r'(?i)(?:^|\n)\s*course\s*:',
    r'(?i)trainee\s+reference',
    r'(?i)lesson\s+plan',
    r'(?i)general\s+order',
    r'(?i)training\s+record',
    r'(?i)memorandum',
]


def is_new_document(text):
    """Check if this page starts a new document based on header patterns."""
    header = text[:500]
    for pattern in DOCUMENT_START_PATTERNS:
        if re.search(pattern, header):
            return True
    return False


def get_department_quick(text):
    """Quick department name extraction for boundary detection."""
    match = re.search(
        r'([A-Z][\w\s]+(?:Police Department|Sheriff[\'\'\u2019]?s?\s*(?:Office|Department)))',
        text[:1500]
    )
    if match:
        dept = re.sub(r'^\d+\s*', '', match.group(1)).strip()
        return re.sub(r'\s+', ' ', dept) if len(dept) > 5 else None
    return None


# ── Group pages into cases ──
cases = []
current_case = []

for i, page in enumerate(all_pages):
    text = page["text"]

    if len(text.strip()) < 30:
        print(f"  Page {page['page_num']:2d}: ⏭️  Skipped (nearly empty)")
        continue

    start_new = False

    if not current_case:
        start_new = True
    elif page["source_file"] != current_case[-1]["source_file"]:
        start_new = True   # Different PDF = always new case
    elif is_new_document(text):
        start_new = True   # New document header detected
    else:
        prev_dept = get_department_quick(current_case[-1]["text"])
        curr_dept = get_department_quick(text)
        if prev_dept and curr_dept and prev_dept != curr_dept:
            start_new = True  # Department changed

    if start_new and current_case:
        cases.append(current_case)
        current_case = []

    current_case.append(page)

if current_case:
    cases.append(current_case)

print(f"\n✅ Document boundary detection complete!")
print(f"{'='*65}")
print(f"Total cases identified: {len(cases)}")
print(f"{'='*65}")
for i, c in enumerate(cases):
    pages = [p["page_num"] for p in c]
    methods = sorted(set(p["method"] for p in c))
    preview = c[0]["text"][:50].replace("\n", " ").strip()
    print(f"  Case {i+1:2d}: Pages {str(pages):15s} [{', '.join(methods):12s}] {preview}...")


✅ Document boundary detection complete!
Total cases identified: 29
  Case  1: Pages [1, 2]          [OCR         ] May 26, 2015  Sheriff Kelley Cradduck Major Nathan...
  Case  2: Pages [3]             [OCR         ] Benton County Sheriff’s Office POLICIES AND PROCED...
  Case  3: Pages [4]             [OCR         ] Vi.  Monthly inspection, to include running the mo...
  Case  4: Pages [5, 6, 7, 8, 9] [OCR         ] I.  Ill.  ARMORED RESCUE VEHICLE (ARV) OPERATIONS...
  Case  5: Pages [10, 11, 12, 13] [OCR         ] Bryant Police Department  312 Roya Lane Bryant, AR...
  Case  6: Pages [14]            [OCR         ] Cabot Police Department MRAP Road Course and Train...
  Case  7: Pages [15]            [OCR         ] vd aa  ya eraiqneddso.ore  May 29, 2015  To Whom I...
  Case  8: Pages [16]            [OCR         ] e Falls, smashed and pinched extremities ¢ Rollove...
  Case  9: Pages [17]            [OCR         ] To: Whom it may Concern From: Craighead County She...
  Case 10: Pag

In [7]:
# =============================================
# CELL 7: Structured Field Extraction from Each Case
# =============================================
# Extracts: Department, Doc_Type, Incident_Type, Date, Program, Key_Detail
# Uses: spaCy NER + Regex + keyword classification


def clean_text(text):
    """Remove page numbers, excess whitespace, artifacts."""
    text = re.sub(r'^\s*\d{1,2}\s*\n', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {3,}', ' ', text)
    return text.strip()


# ────────────────────────────────────────────
# 1. Department Extraction
# ────────────────────────────────────────────
def extract_department(text):
    """Extract police department or sheriff's office name."""
    patterns = [
        r'([A-Z][\w\s]+(?:Police Department))',
        r'([A-Z][\w\s]+(?:Sheriff[\'\'\u2019]?s?\s*(?:Office|Department)))',
        r'From:\s*(?:Lt\.?|Sgt\.?|Cpl\.?|Chief)?\s*[\w\s\.]*?([A-Z][\w\s]+(?:Police|Sheriff))',
    ]
    for pattern in patterns:
        match = re.search(pattern, text[:3000])
        if match:
            dept = match.group(1).strip()
            dept = re.sub(r'^\d+\s*', '', dept).strip()
            dept = re.sub(r'\s+', ' ', dept).strip()
            if len(dept) > 5:
                return dept

    doc = nlp(text[:5000])
    orgs = [ent.text.strip() for ent in doc.ents
            if ent.label_ == "ORG" and len(ent.text.strip()) > 5]
    return orgs[0] if orgs else "Unknown Department"


# ────────────────────────────────────────────
# 2. Doc Type Classification
# ────────────────────────────────────────────
def detect_doc_type(text):
    """Classify document type for this case."""
    text_lower = text[:2000].lower()
    type_map = [
        ("MRAP SOP",                    ["standard operating procedure", "sop"]),
        ("Cover Sheet",                  ["cover sheet"]),
        ("Training Course Summary",      ["course summary"]),
        ("Training Course Outline",      ["course outline"]),
        ("Training Lesson Plan",         ["lesson plan", "lesson title"]),
        ("Training Course",              ["course:", "cleet continuing education"]),
        ("Trainee Reference Materials",  ["trainee reference"]),
        ("MRAP Memorandum",              ["to whom it may concern", "memorandum"]),
        ("General Order / Policy",       ["general order", "policy"]),
        ("Equipment Request",            ["1033 program", "leso", "equipment request"]),
        ("Training Record",              ["training record", "attendance"]),
        ("Invoice / Shipping Record",    ["invoice", "nsn", "ship to", "receipt"]),
        ("MRAP Operations Plan",         ["mrap operation", "vehicle operation"]),
    ]
    for doc_type, keywords in type_map:
        for kw in keywords:
            if kw in text_lower:
                return doc_type
    return "General Report"


# ────────────────────────────────────────────
# 3. Incident Type Classification (AI-style)
# ────────────────────────────────────────────
def detect_incident_type(text):
    """Classify the incident/subject category using keyword matching."""
    text_lower = text.lower()

    incident_keywords = {
        "Tactical / SWAT":     ["swat", "tactical", "ballistic", "armored", "combat",
                                "hostage", "barricade", "active shooter"],
        "Fire / Explosion":    ["fire", "smoke", "burn", "explosion", "arson", "hazmat"],
        "Vehicle Accident":    ["accident", "crash", "collision", "vehicle"],
        "Assault / Violence":  ["assault", "attack", "injured", "violence", "weapon",
                                "shooting", "stabbing", "homicide"],
        "Theft / Robbery":     ["theft", "robbery", "stolen", "burglary", "break-in"],
        "MRAP Operations":     ["mrap", "mine resistant", "ambush protected"],
        "Training / Education":["training", "lesson", "course", "education", "instruction"],
        "Equipment Transfer":  ["equipment", "transfer", "1033", "leso", "surplus"],
    }

    for incident_type, keywords in incident_keywords.items():
        if any(kw in text_lower for kw in keywords):
            return incident_type

    return "Other"


# ────────────────────────────────────────────
# 4. Date Extraction (ZIP-code safe)
# ────────────────────────────────────────────
def extract_date(text):
    """Extract date from document, carefully avoiding ZIP codes."""
    # Priority 1: Labeled date field
    match = re.search(
        r'(?:Date|Effective Date|Dated)[:\s]+'
        r'((?:January|February|March|April|May|June|July|August|September|'
        r'October|November|December)\s+\d{1,2}(?:st|nd|rd|th)?,?\s*\d{4})',
        text, re.IGNORECASE
    )
    if match:
        return match.group(1).strip()

    # Priority 2: Labeled numeric date
    match = re.search(
        r'(?:Date|Effective Date)[:\s]+(\d{1,2}/\d{1,2}/\d{2,4})',
        text, re.IGNORECASE
    )
    if match:
        return match.group(1).strip()

    # Priority 3: Standalone written date (not in address)
    match = re.search(
        r'(?<!AR\s)(?<!,\s)((?:January|February|March|April|May|June|July|'
        r'August|September|October|November|December)\s+\d{1,2}(?:st|nd|rd|th)?'
        r',?\s*\d{4})', text
    )
    if match:
        return match.group(1).strip()

    # Priority 4: spaCy DATE entities (skip ZIP codes)
    doc = nlp(text[:5000])
    for ent in doc.ents:
        if ent.label_ == "DATE" and re.search(r'\d{4}', ent.text):
            if not re.match(r'^[A-Z]{2}\s*\d{5}', ent.text):
                return ent.text.strip()

    return "Unknown"


# ────────────────────────────────────────────
# 5. Officer Extraction (Context-Aware)
# ────────────────────────────────────────────
def extract_officer(text):
    """Extract officer/author name with context awareness.
    Uses positional clues (From:, Prepared by:, rank titles)
    to avoid picking up random person names."""

    # Priority 1: Regex — names after authority labels
    patterns = [
        r'(?:From|Prepared by|Submitted by|Instructor|Point of Contact)'
        r'[,:\s]+\n?\s*(?:Lt\.?|Sgt\.?|Cpl\.?|Chief|Capt\.?|Officer|Deputy)?'
        r'\s*([A-Z][a-z]+\.?\s+[A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)',

        r'(?:Cpl|Sgt|Lt|Capt|Chief|Officer|Deputy|Det)\.?\s+'
        r'([A-Z][a-z]+\s+[A-Z][a-z]+)',

        r'Sheriff\s*[-–—:]?\s*\n?\s*([A-Z][a-z]+\s+[A-Z][a-z]+)',
    ]
    for pattern in patterns:
        match = re.search(pattern, text[:5000])
        if match:
            name = match.group(1).strip()
            # Filter out false positives
            skip_words = ["Police Department", "Sheriff Office", "Unknown",
                          "County Sheriff", "Training Plan"]
            if len(name) > 4 and not any(sw in name for sw in skip_words):
                return name[:60]

    # Priority 2: spaCy PERSON entities — but ONLY near officer-related context
    doc = nlp(text[:5000])
    text_lower = text[:5000].lower()
    officer_context = any(kw in text_lower for kw in
        ["officer", "from:", "prepared by", "submitted by", "instructor",
         "sheriff", "chief", "lieutenant", "sergeant", "corporal", "captain"])

    if officer_context:
        persons = [ent.text.strip() for ent in doc.ents
                   if ent.label_ == "PERSON" and len(ent.text.strip()) > 4]
        if persons:
            return persons[0][:60]

    return "Unknown"


# ────────────────────────────────────────────
# 6. Location Extraction (spaCy NER + Regex)
# ────────────────────────────────────────────
def extract_location(text):
    """Extract location using NER + address patterns."""
    locations = []

    # Structured address patterns
    for match in re.finditer(r'([A-Z][\w\s]+,\s*(?:AR|Arkansas)(?:\s*\d{5})?)', text[:3000]):
        loc = match.group(1).strip()
        if len(loc) > 3 and loc not in locations:
            locations.append(loc)

    # spaCy NER
    doc = nlp(text[:5000])
    for ent in doc.ents:
        if ent.label_ in ["GPE", "LOC", "FAC"] and len(ent.text.strip()) > 2:
            loc = ent.text.strip()
            if loc not in locations:
                locations.append(loc)

    return ", ".join(locations[:3]) if locations else "Arkansas"


# ────────────────────────────────────────────
# 7. Program Classification
# ────────────────────────────────────────────
def assign_program(text):
    """Assign program category."""
    text_lower = text.lower()
    if any(kw in text_lower for kw in ["mrap", "tactical", "weapon", "armored",
                                        "swat", "mine resistant", "1033"]):
        return "Law Enforcement Support"
    elif any(kw in text_lower for kw in ["training", "lesson", "course",
                                          "instruction", "education"]):
        return "Training Program"
    elif any(kw in text_lower for kw in ["vehicle", "aircraft", "equipment transfer"]):
        return "Equipment Transfer"
    return "General"


# ────────────────────────────────────────────
# 8. Key Detail / Summary (Clean, Not Raw)
# ────────────────────────────────────────────
def generate_key_detail(text):
    """Generate a clean, professional summary."""
    # Strip boilerplate headers, addresses, phone/fax numbers
    cleaned = re.sub(
        r'(?:To|From|Date|Ref|Subject|Fax|Office|Phone)[:\s]+[^\n]*\n?',
        '', text[:2000]
    )
    cleaned = re.sub(r'\d{3}[\-.]\d{3}[\-.]\d{4}', '', cleaned)
    cleaned = re.sub(r'www\.\S+', '', cleaned)
    cleaned = re.sub(r'\d+\s+[\w\s]*(?:Lane|Street|Ave|Blvd|Drive)\b', '', cleaned)

    # Normalize whitespace and take first 50 meaningful words
    words = cleaned.split()
    words = [w for w in words if len(w) > 1]  # Drop single-char artifacts

    if len(words) > 10:
        summary = " ".join(words[:50])
        return summary[:300] + ("..." if len(summary) > 300 else "")
    return " ".join(words)[:250] + "..."


# ════════════════════════════════════════════
# PROCESS EACH CASE
# ════════════════════════════════════════════
print("Extracting structured fields from each case...\n")
results = []

for case_idx, case_pages in enumerate(cases):
    combined_text = "\n\n".join(clean_text(p["text"]) for p in case_pages)
    methods_used = sorted(set(p["method"] for p in case_pages))
    page_nums = [p["page_num"] for p in case_pages]
    source = case_pages[0]["source_file"]

    record = {
        "Report_ID":         f"RPT_{case_idx + 1:03d}",
        "Department":        extract_department(combined_text),
        "Doc_Type":          detect_doc_type(combined_text),
        "Incident_Type":     detect_incident_type(combined_text),
        "Date":              extract_date(combined_text),
        "Location":          extract_location(combined_text),
        "Officer":           extract_officer(combined_text),
        "Program":           assign_program(combined_text),
        "Key_Detail":        generate_key_detail(combined_text),
        "Pages":             str(page_nums),
        "Extraction_Method": " + ".join(methods_used),
    }
    results.append(record)

    print(f"  Case {case_idx+1:2d} (pg {str(page_nums):12s}): "
          f"{record['Department'][:25]:25s} | {record['Doc_Type'][:20]:20s} | "
          f"{record['Incident_Type'][:18]:18s} | {record['Date'][:15]:15s}")

print(f"\n{'='*70}")
print(f"✅ Total cases extracted: {len(results)}")
print(f"{'='*70}")
print("\n✅ Processing completed successfully!")

Extracting structured fields from each case...

  Case  1 (pg [1, 2]      ): Benton County Sheriffs Of | MRAP Memorandum      | Fire / Explosion   | May 26, 2015   
  Case  2 (pg [3]         ): Benton County Sheriff’s O | Equipment Request    | Tactical / SWAT    | 8/1/14         
  Case  3 (pg [4]         ): MRAP must be parked in an | General Report       | Tactical / SWAT    | 8/1/14         
  Case  4 (pg [5, 6, 7, 8, 9]): Benton Police Department  | Training Lesson Plan | Tactical / SWAT    | July 1, 2014   
  Case  5 (pg [10, 11, 12, 13]): Bryant Police Department  | General Order / Poli | Tactical / SWAT    | July 2,
2014   
  Case  6 (pg [14]        ): Cabot Police Department   | Training Record      | MRAP Operations    | Unknown        
  Case  7 (pg [15]        ): Craighead County Sheriff' | MRAP Memorandum      | Fire / Explosion   | May 29, 2015   
  Case  8 (pg [16]        ): Chief Deputy Craighead Co | General Report       | Training / Educati | Unknown        
  Case  9

In [8]:
# =============================================
# CELL 8: Build DataFrame & Quality Report
# =============================================

df_documents = pd.DataFrame(results)

print("📊 EXTRACTION QUALITY REPORT")
print("=" * 70)
print(f"Total cases extracted: {len(df_documents)}")
print(f"\n📋 Extraction Methods:")
for method, count in df_documents["Extraction_Method"].value_counts().items():
    print(f"   {method}: {count}")
print(f"\n📄 Document Types (Doc_Type):")
for dtype, count in df_documents["Doc_Type"].value_counts().items():
    print(f"   {dtype}: {count}")
print(f"\n🏢 Departments:")
for dept, count in df_documents["Department"].value_counts().items():
    print(f"   {dept}: {count}")
print(f"\n⚡ Incident Types:")
for itype, count in df_documents["Incident_Type"].value_counts().items():
    print(f"   {itype}: {count}")
print(f"\n📈 Field Extraction Success:")
print(f"   Date:     {(df_documents['Date'] != 'Unknown').sum():>2d}/{len(df_documents)}")
print(f"   Officer:  {(df_documents['Officer'] != 'Unknown').sum():>2d}/{len(df_documents)}")
print(f"   Location: {(df_documents['Location'] != 'Arkansas').sum():>2d}/{len(df_documents)}")
print("=" * 70)

# Preview the key columns
df_documents[["Report_ID", "Department", "Doc_Type", "Date", "Program", "Key_Detail"]]

📊 EXTRACTION QUALITY REPORT
Total cases extracted: 29

📋 Extraction Methods:
   OCR: 18
   pdfplumber: 9
   OCR + pdfplumber: 2

📄 Document Types (Doc_Type):
   Equipment Request: 6
   Training Lesson Plan: 6
   MRAP Memorandum: 4
   General Report: 4
   General Order / Policy: 2
   Training Course Outline: 2
   Training Record: 1
   Cover Sheet: 1
   Training Course Summary: 1
   MRAP SOP: 1
   Trainee Reference Materials: 1

🏢 Departments:
   Hot Springs Police Department: 3
   Lonoke County Sheriff’s Office: 3
   Craighead County Sheriff's Office: 2
   Fort Smith Police Department: 2
   MRAP must be parked in an area designated by the commander at the Benton County Sheriffs Office: 1
   Benton County Sheriff’s Office: 1
   Benton County Sheriffs Office: 1
   Cabot Police Department: 1
   Bryant Police Department: 1
   Benton Police Department: 1
   Chief Deputy Craighead County Sheriff's Office: 1
   Crawford County Sheriff's Office: 1
   CLEET instructor from the Fort Smith Police 

,Report_ID,Department,Doc_Type,Date,Program,Key_Detail
0,RPT_001,Benton County Sheriffs Office,MRAP Memorandum,"May 26, 2015",Law Enforcement Support,"May 26, 2015 Sheriff Kelley Cradduck Major Nat..."
1,RPT_002,Benton County Sheriff’s Office,Equipment Request,8/1/14,Law Enforcement Support,Benton County Sheriff’s SUBJECT MRAP Operation...
2,RPT_003,MRAP must be parked in an area designated by t...,General Report,8/1/14,Law Enforcement Support,"Vi. Monthly inspection, to include running the..."
3,RPT_004,Benton Police Department,Training Lesson Plan,"July 1, 2014",Law Enforcement Support,I. Ill. ARMORED RESCUE VEHICLE (ARV) OPERATION...
4,RPT_005,Bryant Police Department,General Order / Policy,"July 2,\n2014",Law Enforcement Support,"Bryant Police Department Bryant, AR 72022 Mark..."
5,RPT_006,Cabot Police Department,Training Record,Unknown,Law Enforcement Support,Cabot Police Department MRAP Road Course and T...
6,RPT_007,Craighead County Sheriff's Office,MRAP Memorandum,"May 29, 2015",Law Enforcement Support,"vd aa ya eraiqneddso.ore May 29, 2015 As of th..."
7,RPT_008,Chief Deputy Craighead County Sheriff's Office,General Report,Unknown,Training Program,"Falls, smashed and pinched extremities Rollove..."
8,RPT_009,Craighead County Sheriff's Office,Equipment Request,May 2015,Law Enforcement Support,The following documentation is intended to doc...
9,RPT_010,MRAP Craighead County Sheriff's Office,Training Lesson Plan,Unknown,Law Enforcement Support,COURSE: LESSON TITLE: CLEST Continuing Educati...


In [9]:
# =============================================
# CELL 9: Generate Final Structured CSV
# =============================================
# Expected output columns: Report_ID, Department, Doc_Type, Date, Program, Key_Detail
# + Incident_ID for integration readiness

df_output = df_documents[[
    "Report_ID", "Department", "Doc_Type", "Date", "Program", "Key_Detail"
]].copy()

# Add Incident_ID for integration (IMPORTANT for final merge step)
df_output.insert(0, "Incident_ID", ["INC_" + str(i+1).zfill(3) for i in range(len(df_output))])

# Save final CSV
df_output.to_csv("outputs/document_analyst_output.csv", index=False)

print("=" * 70)
print("📄 DOCUMENT ANALYST — FINAL OUTPUT")
print("=" * 70)
print(f"Total cases:  {len(df_output)}")
print(f"Columns:      {list(df_output.columns)}")
print(f"Saved to:     outputs/document_analyst_output.csv")
print("=" * 70)
print()
print("✅ Document Analyst processing completed successfully!")
print()
df_output

📄 DOCUMENT ANALYST — FINAL OUTPUT
Total cases:  29
Columns:      ['Incident_ID', 'Report_ID', 'Department', 'Doc_Type', 'Date', 'Program', 'Key_Detail']
Saved to:     outputs/document_analyst_output.csv

✅ Document Analyst processing completed successfully!



,Incident_ID,Report_ID,Department,Doc_Type,Date,Program,Key_Detail
0,INC_001,RPT_001,Benton County Sheriffs Office,MRAP Memorandum,"May 26, 2015",Law Enforcement Support,"May 26, 2015 Sheriff Kelley Cradduck Major Nat..."
1,INC_002,RPT_002,Benton County Sheriff’s Office,Equipment Request,8/1/14,Law Enforcement Support,Benton County Sheriff’s SUBJECT MRAP Operation...
2,INC_003,RPT_003,MRAP must be parked in an area designated by t...,General Report,8/1/14,Law Enforcement Support,"Vi. Monthly inspection, to include running the..."
3,INC_004,RPT_004,Benton Police Department,Training Lesson Plan,"July 1, 2014",Law Enforcement Support,I. Ill. ARMORED RESCUE VEHICLE (ARV) OPERATION...
4,INC_005,RPT_005,Bryant Police Department,General Order / Policy,"July 2,\n2014",Law Enforcement Support,"Bryant Police Department Bryant, AR 72022 Mark..."
5,INC_006,RPT_006,Cabot Police Department,Training Record,Unknown,Law Enforcement Support,Cabot Police Department MRAP Road Course and T...
6,INC_007,RPT_007,Craighead County Sheriff's Office,MRAP Memorandum,"May 29, 2015",Law Enforcement Support,"vd aa ya eraiqneddso.ore May 29, 2015 As of th..."
7,INC_008,RPT_008,Chief Deputy Craighead County Sheriff's Office,General Report,Unknown,Training Program,"Falls, smashed and pinched extremities Rollove..."
8,INC_009,RPT_009,Craighead County Sheriff's Office,Equipment Request,May 2015,Law Enforcement Support,The following documentation is intended to doc...
9,INC_010,RPT_010,MRAP Craighead County Sheriff's Office,Training Lesson Plan,Unknown,Law Enforcement Support,COURSE: LESSON TITLE: CLEST Continuing Educati...


### ✅ Document Analyst Complete!
**Output:** `outputs/document_analyst_output.csv`

### Output Schema:
| Column | Description |
|--------|-------------|
| Incident_ID | Integration-ready ID (INC_001, INC_002, ...) for final merge |
| Report_ID | Unique case identifier (RPT_001, RPT_002, ...) |
| Department | Police department or sheriff's office name |
| Doc_Type | Document classification (MRAP Memorandum, SOP, Training Plan, Cover Sheet, etc.) |
| Date | Extracted report date |
| Program | Associated program (Law Enforcement Support, Training Program, etc.) |
| Key_Detail | Clean summary of key findings (boilerplate removed) |

### Tools Used:
| Tool | Purpose |
|------|---------|
| **pdfplumber** | Primary text extraction (structured PDFs) |
| **PyMuPDF (fitz)** | Secondary text extraction + page rendering for OCR |
| **pytesseract** | OCR for scanned pages (auto-detected when text < 50 chars) |
| **spaCy (en_core_web_sm)** | NER for locations, persons, organizations, dates |
| **Regex** | Dates, department names, document boundaries, officer names |

### AI Techniques:
- **Keyword-based incident type classification** — maps document content to categories (Tactical/SWAT, Fire, Assault, MRAP Operations, Training, etc.)
- **Document boundary detection** — pattern matching on headers to identify where new cases begin
- **Context-aware officer extraction** — uses positional clues (From:, Prepared by:, rank titles) instead of blindly picking any PERSON entity
- **Smart OCR fallback** — auto-detects scanned pages and applies 300 DPI OCR only when needed
